In [1]:
import os
import time
import cv2
import numpy as np
import tensorflow as tf
from pathlib import Path
import pandas as pd

In [2]:
DATASET_DIR = Path("/kaggle/input/datasets/joy1008/projectdetectionframes/deepfake_frames")

TRAIN_DIR = DATASET_DIR/"train"
VAL_DIR = DATASET_DIR/"val"
TEST_DIR = DATASET_DIR/"test"

print("Train: ", TRAIN_DIR)
print("Validation: ", VAL_DIR)
print("Test: ", TEST_DIR)

Train:  /kaggle/input/datasets/joy1008/projectdetectionframes/deepfake_frames/train
Validation:  /kaggle/input/datasets/joy1008/projectdetectionframes/deepfake_frames/val
Test:  /kaggle/input/datasets/joy1008/projectdetectionframes/deepfake_frames/test


In [3]:
IMAGE_SIZE = (299, 299)

def load_and_resize(image_path):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    return image

In [4]:
data_augmentation = tf.keras.Sequential([tf.keras.layers.RandomFlip("horizontal"),
                                        tf.keras.layers.RandomRotation(0.05),
                                        tf.keras.layers.RandomZoom(0.10),
                                        tf.keras.layers.RandomTranslation(height_factor=0.05, width_factor=0.05)])

I0000 00:00:1790876177.207122      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790876177.210608      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [5]:
def get_image_paths_and_labels(folder):
    folder = Path(folder)

    image_paths =[]
    labels =[]

    for image_path in folder.rglob("*.jpg"):
        image_paths.append(str(image_path))

        if image_path.parent.parent.name == "real":
            labels.append(0)

        elif image_path.parent.parent.name == "fake":
            labels.append(1)

    return image_paths, labels

In [6]:
train_paths, train_labels = get_image_paths_and_labels(TRAIN_DIR)
val_paths, val_labels = get_image_paths_and_labels(VAL_DIR)
test_paths, test_labels = get_image_paths_and_labels(TEST_DIR)

print("Train: ", len(train_paths))
print("Validation: ", len(val_paths))
print("Test: ", len(test_paths))

print("Example path: ", train_paths[0])
print("Example label: ", train_labels[0])

Train:  49000
Validation:  10500
Test:  10500
Example path:  /kaggle/input/datasets/joy1008/projectdetectionframes/deepfake_frames/train/fake/Face2Face_200_189/frame_001.jpg
Example label:  1


In [7]:
def scan(folder, split):
    rows = []
    for root, dirs, files in os.walk(folder):
        label = 0 if Path(root).parent.name == "real" else 1
        for f in files:
            if f.endswith(".jpg"):
                rows.append((os.path.join(root, f), label, split))
    return rows

start = time.time()
rows = scan(TRAIN_DIR, "train") + scan(VAL_DIR, "val") + scan(TEST_DIR, "test")

df = pd.DataFrame(rows, columns=["path", "label", "split"])
df.to_csv("/kaggle/working/manifest.csv", index=False)

print("Saved", len(df), "rows in", round(time.time() - start), "seconds")
print(df.groupby(["split", "label"]).size())

Saved 70000 rows in 6 seconds
split  label
test   0         1500
       1         9000
train  0         7000
       1        42000
val    0         1500
       1         9000
dtype: int64
